# 00 - Setup and Validation

This notebook is the shared prerequisite check for **all** demos in the AI
Governance workshop. Run it first, top to bottom, before opening any of the
`demoN-*.ipynb` notebooks.

It will:

1. Confirm you are logged in via `az login` and show the active subscription.
2. Let you choose/confirm the Azure subscription to use.
3. Prompt for (and persist to `.env`) the resource group and APIM instance
   name that the rest of the workshop will use.
4. Verify the APIM instance is reachable and print its gateway URL and SKU.

Nothing here creates or modifies any Azure resources -- it only reads
configuration and validates connectivity.


In [ ]:
import sys
sys.path.append("..")

from shared import auth, config, display


## 1. Confirm Azure CLI login

In [ ]:
import subprocess

try:
    result = subprocess.run(
        ["az", "account", "show", "-o", "json"],
        capture_output=True, text=True, check=True, timeout=30,
    )
    display.banner("az login is active.", kind="success")
    print(result.stdout)
except Exception as exc:
    display.banner(
        "Could not confirm az login. Run `az login` in a terminal, then re-run this cell.",
        kind="error",
    )
    raise


## 2. Load or collect workshop configuration

In [ ]:
cfg = config.load_config(interactive=True)
config.validate_config(cfg)
display.header("Current configuration (secrets masked)")
display.show_table([cfg.as_display_dict()])


## 3. Validate the APIM instance is reachable

In [ ]:
from shared import apim

service = apim.get_service(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
gateway_url = service["properties"]["gatewayUrl"]
sku = service.get("sku", {}).get("name", "unknown")

display.banner(f"APIM instance '{cfg.apim_name}' is reachable.", kind="success")
display.show_table([
    {
        "name": service.get("name"),
        "location": service.get("location"),
        "sku": sku,
        "gatewayUrl": gateway_url,
        "provisioningState": service.get("properties", {}).get("provisioningState"),
    }
])

if sku in ("Consumption", "Developer") is False and sku not in (
    "Premium", "StandardV2", "PremiumV2", "Developer", "Standard",
):
    display.banner(
        "Note: the azure-openai-token-limit policy used in Demo 1 requires a "
        "supported APIM tier (Standard/Premium classic tiers or the StandardV2/"
        "PremiumV2 v2 tiers). Verify your SKU supports it before running Demo 1.",
        kind="warning",
    )


## Next steps

Configuration has been captured and persisted to `.env`. You can now open:

- `demo1-token-limits.ipynb` -- Token limits & quota enforcement (complete)
- `demo2-placeholder.ipynb` -- coming soon
- `demo3-placeholder.ipynb` -- coming soon
- `demo4-placeholder.ipynb` -- coming soon
